<div align="center">

# WEEK 03
## A4 FEATURE ENGINEERING FOR ANALYTICS

### CadetX Virtual Work Experience
### Heavy Supplier, Inventory & Warehouse Analytics

</div>

---

| | |
|---|---|
| **Prepared by** | Keren Oyem (Data Analyst / Data Scientist) |
| **Sprint** | Week 3 · Fri 6 Nov – Thu 12 Nov 2026 |
| **Catalogue task** | A4 Feature Engineering for Analytics |
| **Input** | Schema `star` in `warehouse.duckdb` (built in `03_star_schema.ipynb`) |
| **Output** | Schema `feat` (8 feature tables) and the reusable library `src/features.py` |
| **Tools** | Python, SQL on DuckDB |

## Purpose

The star schema holds clean facts, but analysis needs **derived measures**: how late was a supplier, how long does a customer take to pay, how variable is demand. This notebook builds those measures once, documents them, and stores them so Weeks 4-12 (KPIs, segmentation, forecasting, optimisation) reuse the same definitions.

## Features built (from the Week 3 plan)

| Plan item | Feature table |
|---|---|
| Actual supplier lead time, delay days, on-time flag | `feat_purchase_order`, `feat_supplier` |
| Order-to-delivery days | `feat_sales_order` |
| Days-to-pay, late and overdue flags | `feat_invoice` |
| Credit utilisation, recency, tenure | `feat_customer` |
| Unit volume / weight, margin values | `feat_product` |
| Monthly demand per product-branch, calendar features, lags | `feat_monthly_demand` |
| Demand CV and demand pattern | `feat_demand_variability` |

## How this notebook works

1. The code cell in section 1 **writes `src/features.py`** (the feature library, one SQL statement per table, with a description for every table and column).
2. Section 2 imports it and builds all tables.
3. Sections 3-4 inspect each feature and validate it.

## Rules carried from Week 2

Cancelled orders and POs are kept but excluded from delay, lead-time and delivery features. Invoices are keyed on `so_id`. Ambiguous invoices (re-issued `invoice_id`) get no payment-timing features. Reference date for recency and overdue measures: **31 Dec 2024**.

## 0. Setup

**Objective.** Open the database and confirm the star schema exists.

Run Jupyter from the **project folder** (contains `warehouse.duckdb`, `sql/`, `src/`, `data/processed/`).

In [1]:
import importlib, os, sys
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

try:
    con.close()
except Exception:
    pass

con = duckdb.connect("warehouse.duckdb")
n_star = con.execute("SELECT COUNT(*) FROM duckdb_tables() WHERE schema_name = 'star'").fetchone()[0]
assert n_star == 9, "Schema `star` not found. Run 03_star_schema.ipynb first."
os.makedirs("src", exist_ok=True)
print("Connected. Star schema found with", n_star, "tables.")

Connected. Star schema found with 9 tables.


**Reading the output.** `Connected. Star schema found with 9 tables.` If the assertion fails, run `03_star_schema.ipynb` first.

## 1. Write the feature library (`src/features.py`)

**Objective.** Save the feature definitions as a reusable Python module.

**Why a module?** The plan asks for a *feature library with docstrings*. Keeping the SQL in one file means the notebook, later notebooks and automation all build features the same way.

**What the file contains**

| Part | Meaning |
|---|---|
| `FEATURE_SQL` | One `CREATE TABLE ... AS SELECT` per feature table |
| `TABLE_DESCRIPTIONS`, `COLUMN_DESCRIPTIONS` | A description for every table and column (stored in the database, used later by the data dictionary) |
| `build_features(con)` | Builds every table, attaches descriptions, returns a summary |
| `undocumented_columns(con)` | Lists any column without a description (expected: none) |

The `%%writefile` line on the first row tells Jupyter to save the cell as a file instead of running it.

In [2]:
%%writefile src/features.py
"""Week 3 (A4): feature library. Builds the `feat` schema from the `star` schema.

Usage (from the project folder that contains warehouse.duckdb):

    import duckdb
    from src.features import build_features
    con = duckdb.connect("warehouse.duckdb")
    summary = build_features(con)

Each feature table is one SQL statement (FEATURE_SQL), documented by
TABLE_DESCRIPTIONS and COLUMN_DESCRIPTIONS. Descriptions are stored inside the
database (COMMENT ON) and feed the data dictionary (A6).

Rules carried from Week 2: cancelled orders/POs are kept but excluded from
delay, lead-time and delivery features; invoices are keyed on so_id;
ambiguous invoices (re-issued invoice_id) get no payment-timing features.
"""
REF_DATE = "2024-12-31"        # reference date for recency / overdue measures
DEMAND_START = "2019-01-01"    # monthly demand grid: Jan 2019 ...
DEMAND_END = "2024-12-31"      # ... to Dec 2024

FEATURE_SQL = {}

# --------------------------------------------------------------------------
FEATURE_SQL["feat_purchase_order"] = """
CREATE OR REPLACE TABLE feat.feat_purchase_order AS
WITH po AS (
    SELECT po_id, supplier_id, branch_id, order_date_key, expected_date_key,
           received_date_key, po_status, is_cancelled,
           COUNT(*)              AS line_count,
           SUM(quantity)         AS units,
           SUM(line_total)       AS spend_before_gst,
           SUM(line_grand_total) AS spend_incl_gst
    FROM star.fact_purchase_line
    GROUP BY ALL
)
SELECT
    po.po_id, po.supplier_id, po.branch_id,
    od.full_date          AS order_date,
    ed.full_date          AS expected_date,
    rd.full_date          AS received_date,
    od.calendar_year      AS order_year,
    po.po_status, po.is_cancelled, po.line_count, po.units,
    po.spend_before_gst, po.spend_incl_gst,
    date_diff('day', od.full_date, ed.full_date) AS planned_lead_time_days,
    date_diff('day', od.full_date, rd.full_date) AS actual_lead_time_days,
    date_diff('day', ed.full_date, rd.full_date) AS delay_days,
    CASE WHEN po.is_cancelled THEN NULL
         WHEN rd.full_date <= ed.full_date THEN TRUE ELSE FALSE END AS is_on_time,
    sup.lead_time_days AS stated_lead_time_days,
    date_diff('day', od.full_date, rd.full_date) - sup.lead_time_days AS lead_time_gap_days
FROM po
JOIN star.dim_date od ON po.order_date_key = od.date_key
JOIN star.dim_date ed ON po.expected_date_key = ed.date_key
LEFT JOIN star.dim_date rd ON po.received_date_key = rd.date_key
JOIN star.dim_supplier sup ON po.supplier_id = sup.supplier_id
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_supplier"] = """
CREATE OR REPLACE TABLE feat.feat_supplier AS
WITH agg AS (
    SELECT s.supplier_id, s.supplier_name,
           s.lead_time_days AS stated_lead_time_days,
           s.reliability_score AS stated_reliability_score,
           s.import_duty_rate,
           COUNT(p.po_id)                                  AS pos_placed,
           COUNT(*) FILTER (WHERE p.is_cancelled)          AS pos_cancelled,
           COUNT(*) FILTER (WHERE NOT p.is_cancelled)      AS pos_received,
           AVG(p.actual_lead_time_days)                    AS avg_actual_lead_time_days,
           STDDEV_SAMP(p.actual_lead_time_days)            AS lead_time_std_days,
           AVG(p.delay_days)                               AS avg_delay_days,
           quantile_cont(p.delay_days, 0.9)                AS p90_delay_days,
           AVG(p.is_on_time::INT)                          AS on_time_rate,
           SUM(p.spend_before_gst) FILTER (WHERE NOT p.is_cancelled) AS spend_before_gst
    FROM star.dim_supplier s
    LEFT JOIN feat.feat_purchase_order p ON s.supplier_id = p.supplier_id
    GROUP BY s.supplier_id, s.supplier_name, s.lead_time_days,
             s.reliability_score, s.import_duty_rate
)
SELECT supplier_id, supplier_name, stated_lead_time_days, stated_reliability_score,
       import_duty_rate, pos_placed, pos_cancelled,
       pos_cancelled * 1.0 / NULLIF(pos_placed, 0) AS cancellation_rate,
       pos_received, avg_actual_lead_time_days, lead_time_std_days,
       avg_delay_days, p90_delay_days, on_time_rate, spend_before_gst,
       spend_before_gst / SUM(spend_before_gst) OVER () AS spend_share
FROM agg
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_sales_order"] = """
CREATE OR REPLACE TABLE feat.feat_sales_order AS
WITH o AS (
    SELECT so_id, customer_id, branch_id, order_date_key, order_status,
           is_cancelled, sales_channel, payment_terms, days_to_deliver,
           COUNT(*)              AS line_count,
           SUM(quantity)         AS units,
           SUM(line_total)       AS revenue_before_gst,
           SUM(line_grand_total) AS revenue_incl_gst,
           SUM(gross_margin)     AS gross_margin
    FROM star.fact_sales_line
    GROUP BY ALL
)
SELECT o.so_id, o.customer_id, o.branch_id, o.order_status, o.is_cancelled,
       o.sales_channel, o.payment_terms, o.line_count, o.units,
       o.revenue_before_gst, o.revenue_incl_gst, o.gross_margin,
       d.full_date AS order_date, d.calendar_year AS order_year,
       d.calendar_month AS order_month,
       CASE WHEN o.is_cancelled THEN NULL ELSE o.days_to_deliver END AS order_to_delivery_days
FROM o
JOIN star.dim_date d ON o.order_date_key = d.date_key
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_invoice"] = """
CREATE OR REPLACE TABLE feat.feat_invoice AS
WITH base AS (
    SELECT f.so_id, f.invoice_id, f.customer_id, f.branch_id,
           i.full_date AS invoice_date, u.full_date AS due_date,
           i.calendar_year AS invoice_year,
           f.grand_total, f.total_paid, f.outstanding_balance,
           f.payment_status_derived, f.invoice_id_duplicated,
           fp.full_date AS first_payment_date, lp.full_date AS last_payment_date
    FROM star.fact_invoice_payment f
    JOIN star.dim_date i ON f.invoice_date_key = i.date_key
    JOIN star.dim_date u ON f.due_date_key = u.date_key
    LEFT JOIN star.dim_date fp ON f.first_payment_date_key = fp.date_key
    LEFT JOIN star.dim_date lp ON f.last_payment_date_key = lp.date_key
), flagged AS (
    SELECT *,
           (NOT invoice_id_duplicated AND outstanding_balance > 0.01
            AND due_date < DATE '__REF__') AS is_overdue
    FROM base
)
SELECT so_id, invoice_id, customer_id, branch_id, invoice_date, due_date,
       invoice_year, grand_total, total_paid, outstanding_balance,
       payment_status_derived, invoice_id_duplicated,
       date_diff('day', invoice_date, due_date)          AS credit_days,
       date_diff('day', invoice_date, first_payment_date) AS days_to_first_payment,
       date_diff('day', invoice_date, last_payment_date)  AS days_to_last_payment,
       CASE WHEN payment_status_derived = 'Fully paid'
            THEN date_diff('day', invoice_date, last_payment_date) END AS days_to_pay,
       CASE WHEN payment_status_derived = 'Fully paid'
            THEN last_payment_date > due_date END                      AS is_paid_late,
       CASE WHEN payment_status_derived = 'Fully paid'
            THEN GREATEST(date_diff('day', due_date, last_payment_date), 0) END AS days_late_when_paid,
       is_overdue,
       CASE WHEN is_overdue THEN date_diff('day', due_date, DATE '__REF__') END AS days_overdue
FROM flagged
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_customer"] = """
CREATE OR REPLACE TABLE feat.feat_customer AS
WITH s AS (
    SELECT customer_id,
           COUNT(*)                                          AS orders_placed,
           COUNT(*) FILTER (WHERE NOT is_cancelled)          AS delivered_orders,
           SUM(revenue_before_gst) FILTER (WHERE NOT is_cancelled) AS revenue_before_gst,
           MIN(order_date) FILTER (WHERE NOT is_cancelled)   AS first_order_date,
           MAX(order_date) FILTER (WHERE NOT is_cancelled)   AS last_order_date
    FROM feat.feat_sales_order
    GROUP BY customer_id
), pay AS (
    SELECT customer_id,
           AVG(days_to_pay) AS avg_days_to_pay,
           AVG(CASE WHEN is_paid_late THEN 1.0 ELSE 0.0 END)
               FILTER (WHERE is_paid_late IS NOT NULL) AS late_payment_rate
    FROM feat.feat_invoice
    GROUP BY customer_id
)
SELECT c.customer_id, c.customer_type, c.industry_segment, c.region, c.branch_id,
       c.customer_since, c.credit_limit, c.current_balance,
       c.current_balance * 1.0 / NULLIF(c.credit_limit, 0) AS credit_utilisation,
       COALESCE(s.orders_placed, 0)     AS orders_placed,
       COALESCE(s.delivered_orders, 0)  AS delivered_orders,
       COALESCE(s.revenue_before_gst, 0) AS revenue_before_gst,
       s.revenue_before_gst / NULLIF(s.delivered_orders, 0) AS avg_order_value,
       s.first_order_date, s.last_order_date,
       date_diff('day', s.last_order_date, DATE '__REF__') AS recency_days,
       date_diff('day', c.customer_since, DATE '__REF__') / 365.25 AS tenure_years,
       pay.avg_days_to_pay, pay.late_payment_rate
FROM star.dim_customer c
LEFT JOIN s   ON c.customer_id = s.customer_id
LEFT JOIN pay ON c.customer_id = pay.customer_id
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_product"] = """
CREATE OR REPLACE TABLE feat.feat_product AS
SELECT product_id, product_name, category, machine_type, brand, criticality_level,
       unit_cost, unit_price,
       unit_price - unit_cost AS unit_margin,
       (unit_price - unit_cost) * 100.0 / NULLIF(unit_price, 0) AS margin_pct_calc,
       (unit_price - unit_cost) * 100.0 / NULLIF(unit_cost, 0)  AS markup_pct,
       margin_percentage AS margin_pct_stored,
       weight_kg, length_cm, width_cm, height_cm, volume_m3,
       weight_kg / NULLIF(volume_m3, 0)  AS density_kg_per_m3,
       unit_price / NULLIF(weight_kg, 0) AS price_per_kg,
       lead_time_days AS master_lead_time_days
FROM star.dim_product
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_monthly_demand"] = """
CREATE OR REPLACE TABLE feat.feat_monthly_demand AS
WITH months AS (
    SELECT DISTINCT CAST(date_trunc('month', full_date) AS DATE) AS month_start
    FROM star.dim_date
    WHERE full_date BETWEEN DATE '__DSTART__' AND DATE '__DEND__'
), grid AS (
    SELECT p.product_id, b.branch_id, m.month_start
    FROM star.dim_product p
    CROSS JOIN star.dim_branch b
    CROSS JOIN months m
), act AS (
    SELECT f.product_id, f.branch_id,
           CAST(date_trunc('month', d.full_date) AS DATE) AS month_start,
           SUM(f.quantity)        AS units_sold,
           SUM(f.line_total)      AS revenue,
           COUNT(*)               AS order_lines,
           COUNT(DISTINCT f.so_id) AS orders
    FROM star.fact_sales_line f
    JOIN star.dim_date d ON f.order_date_key = d.date_key
    WHERE NOT f.is_cancelled
    GROUP BY ALL
), base AS (
    SELECT g.product_id, g.branch_id, g.month_start,
           COALESCE(a.units_sold, 0)  AS units_sold,
           COALESCE(a.revenue, 0)     AS revenue,
           COALESCE(a.order_lines, 0) AS order_lines,
           COALESCE(a.orders, 0)      AS orders
    FROM grid g
    LEFT JOIN act a ON g.product_id = a.product_id
                   AND g.branch_id = a.branch_id
                   AND g.month_start = a.month_start
)
SELECT product_id, branch_id, month_start,
       year(month_start)    AS calendar_year,
       month(month_start)   AS calendar_month,
       quarter(month_start) AS calendar_quarter,
       CASE WHEN month(month_start) >= 4 THEN year(month_start) + 1
            ELSE year(month_start) END AS fiscal_year,
       day(last_day(month_start)) AS days_in_month,
       units_sold, revenue, order_lines, orders,
       LAG(units_sold, 1)  OVER w AS units_lag_1,
       LAG(units_sold, 12) OVER w AS units_lag_12,
       AVG(units_sold) OVER (PARTITION BY product_id, branch_id
                             ORDER BY month_start
                             ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING) AS units_roll_mean_3
FROM base
WINDOW w AS (PARTITION BY product_id, branch_id ORDER BY month_start)
"""

# --------------------------------------------------------------------------
FEATURE_SQL["feat_demand_variability"] = """
CREATE OR REPLACE TABLE feat.feat_demand_variability AS
WITH s AS (
    SELECT product_id, branch_id,
           COUNT(*)                                   AS months_observed,
           SUM(units_sold)                            AS total_units,
           AVG(units_sold)                            AS mean_monthly_units,
           STDDEV_SAMP(units_sold)                    AS std_monthly_units,
           COUNT(*) FILTER (WHERE units_sold > 0)     AS demand_months,
           COUNT(*) FILTER (WHERE units_sold = 0)     AS zero_demand_months,
           AVG(units_sold) FILTER (WHERE units_sold > 0)         AS mean_nonzero_units,
           STDDEV_SAMP(units_sold) FILTER (WHERE units_sold > 0) AS std_nonzero_units
    FROM feat.feat_monthly_demand
    GROUP BY product_id, branch_id
), m AS (
    SELECT *,
           std_monthly_units / NULLIF(mean_monthly_units, 0)    AS demand_cv,
           zero_demand_months * 1.0 / months_observed           AS zero_demand_share,
           months_observed * 1.0 / NULLIF(demand_months, 0)     AS adi,
           COALESCE(POWER(std_nonzero_units / NULLIF(mean_nonzero_units, 0), 2), 0) AS cv2_nonzero
    FROM s
)
SELECT product_id, branch_id, months_observed, total_units, mean_monthly_units,
       std_monthly_units, demand_cv, demand_months, zero_demand_months,
       zero_demand_share, adi, cv2_nonzero,
       CASE WHEN demand_months = 0              THEN 'No demand'
            WHEN adi < 1.32 AND cv2_nonzero < 0.49  THEN 'Smooth'
            WHEN adi < 1.32                     THEN 'Erratic'
            WHEN cv2_nonzero < 0.49             THEN 'Intermittent'
            ELSE 'Lumpy' END AS demand_class
FROM m
"""

BUILD_ORDER = ["feat_purchase_order", "feat_supplier", "feat_sales_order", "feat_invoice",
               "feat_customer", "feat_product", "feat_monthly_demand",
               "feat_demand_variability"]

TABLE_DESCRIPTIONS = {
    "feat_purchase_order": "Grain: one purchase order (24,000). Planned vs actual lead time, delay and on-time flag. Delay fields are NULL for cancelled POs.",
    "feat_supplier": "Grain: one supplier (8). Stated vs actual performance: lead time, delay, on-time rate, cancellation, spend share.",
    "feat_sales_order": "Grain: one sales order (20,000). Order totals and order-to-delivery days. Delivery days are NULL for cancelled orders.",
    "feat_invoice": "Grain: one invoice (so_id, 18,033). Payment timing, late and overdue flags. Payment timing is NULL for ambiguous invoices.",
    "feat_customer": "Grain: one customer (500). Credit utilisation, order value, recency, tenure and payment behaviour.",
    "feat_product": "Grain: one product (30). Margin, markup, volume, weight and density measures.",
    "feat_monthly_demand": "Grain: product x branch x month (30 x 6 x 72 = 12,960). Delivered sales per month with zero months filled, calendar fields and lag features.",
    "feat_demand_variability": "Grain: product x branch (180). Demand variability (CV), intermittency (ADI) and demand pattern class.",
}

COLUMN_DESCRIPTIONS = {
 "feat_purchase_order": {
  "po_id": "Purchase order ID.", "supplier_id": "FK to dim_supplier.", "branch_id": "FK to dim_branch (receiving branch).",
  "order_date": "Date the PO was placed.", "expected_date": "Promised delivery date.",
  "received_date": "Actual receipt date. NULL for cancelled POs.", "order_year": "Calendar year of the order date.",
  "po_status": "Received or Cancelled.", "is_cancelled": "TRUE for cancelled POs; exclude from delay and lead-time measures.",
  "line_count": "Number of lines on the PO.", "units": "Total units ordered.",
  "spend_before_gst": "PO value before GST, INR.", "spend_incl_gst": "PO value including GST, INR.",
  "planned_lead_time_days": "expected_date - order_date.", "actual_lead_time_days": "received_date - order_date. NULL if cancelled.",
  "delay_days": "received_date - expected_date. Positive = late, negative = early. NULL if cancelled.",
  "is_on_time": "TRUE if received on or before the expected date. NULL if cancelled.",
  "stated_lead_time_days": "Lead time promised in the supplier master.",
  "lead_time_gap_days": "actual_lead_time_days - stated_lead_time_days."},
 "feat_supplier": {
  "supplier_id": "Supplier ID.", "supplier_name": "Supplier name.",
  "stated_lead_time_days": "Lead time stated in the supplier master.", "stated_reliability_score": "Reliability score stated in the supplier master.",
  "import_duty_rate": "Import duty rate from the supplier master.", "pos_placed": "All POs placed with the supplier.",
  "pos_cancelled": "Cancelled POs.", "cancellation_rate": "pos_cancelled / pos_placed (0-1).", "pos_received": "Non-cancelled POs.",
  "avg_actual_lead_time_days": "Mean order-to-receipt days.", "lead_time_std_days": "Standard deviation of actual lead time (variability).",
  "avg_delay_days": "Mean delay vs expected date.", "p90_delay_days": "90th percentile of delay days.",
  "on_time_rate": "Share of received POs on time (0-1).", "spend_before_gst": "Spend on non-cancelled POs, INR before GST.",
  "spend_share": "Supplier share of total spend (0-1)."},
 "feat_sales_order": {
  "so_id": "Sales order ID.", "customer_id": "FK to dim_customer.", "branch_id": "FK to dim_branch.",
  "order_status": "Delivered or Cancelled.", "is_cancelled": "TRUE for cancelled orders.", "sales_channel": "Sales channel.",
  "payment_terms": "Payment terms.", "line_count": "Lines on the order.", "units": "Units on the order.",
  "revenue_before_gst": "Order value before GST, INR.", "revenue_incl_gst": "Order value including GST, INR.",
  "gross_margin": "Order value minus cost at master unit cost, INR.", "order_date": "Order date.",
  "order_year": "Calendar year of the order.", "order_month": "Calendar month of the order (1-12).",
  "order_to_delivery_days": "delivery date - order date. NULL for cancelled orders (placeholder date)."},
 "feat_invoice": {
  "so_id": "Sales order ID; the reliable invoice key.", "invoice_id": "Invoice ID (not unique).", "customer_id": "FK to dim_customer.",
  "branch_id": "FK to dim_branch.", "invoice_date": "Invoice date.", "due_date": "Payment due date.", "invoice_year": "Calendar year of the invoice.",
  "grand_total": "Invoice value incl. GST, INR.", "total_paid": "Payments received, INR.", "outstanding_balance": "grand_total - total_paid. Negative = overpaid.",
  "payment_status_derived": "Status derived from payments.", "invoice_id_duplicated": "TRUE if invoice_id is shared; payment timing is not computed.",
  "credit_days": "due_date - invoice_date.", "days_to_first_payment": "First payment date - invoice date.",
  "days_to_last_payment": "Last payment date - invoice date.", "days_to_pay": "Days from invoice to last payment, only for Fully paid invoices.",
  "is_paid_late": "TRUE if a fully paid invoice was settled after the due date.", "days_late_when_paid": "Days after due date at final payment (0 if on time).",
  "is_overdue": "TRUE if still unpaid at the reference date (2024-12-31) and past due.", "days_overdue": "Reference date - due date for overdue invoices."},
 "feat_customer": {
  "customer_id": "Customer ID.", "customer_type": "Customer type.", "industry_segment": "Industry segment.", "region": "Customer region.",
  "branch_id": "Serving branch.", "customer_since": "Date the customer joined.", "credit_limit": "Credit limit, INR.",
  "current_balance": "Current balance, INR.", "credit_utilisation": "current_balance / credit_limit.",
  "orders_placed": "All orders including cancelled.", "delivered_orders": "Delivered orders.", "revenue_before_gst": "Delivered revenue before GST, INR.",
  "avg_order_value": "revenue_before_gst / delivered_orders.", "first_order_date": "First delivered order date.", "last_order_date": "Last delivered order date.",
  "recency_days": "Reference date (2024-12-31) minus last delivered order.", "tenure_years": "Years from customer_since to the reference date.",
  "avg_days_to_pay": "Mean days to pay on fully paid invoices.", "late_payment_rate": "Share of fully paid invoices settled late (0-1)."},
 "feat_product": {
  "product_id": "Product ID.", "product_name": "Product name.", "category": "Category.", "machine_type": "Machine type.", "brand": "Brand.",
  "criticality_level": "Criticality level.", "unit_cost": "Unit cost, INR.", "unit_price": "Unit price, INR.", "unit_margin": "unit_price - unit_cost.",
  "margin_pct_calc": "(price - cost) / price x 100, recalculated.", "markup_pct": "(price - cost) / cost x 100.",
  "margin_pct_stored": "Margin percentage as stored in the product master.", "weight_kg": "Unit weight, kg.", "length_cm": "Length, cm.",
  "width_cm": "Width, cm.", "height_cm": "Height, cm.", "volume_m3": "Unit volume, cubic metres.", "density_kg_per_m3": "weight_kg / volume_m3.",
  "price_per_kg": "unit_price / weight_kg.", "master_lead_time_days": "Lead time in the product master."},
 "feat_monthly_demand": {
  "product_id": "FK to dim_product.", "branch_id": "FK to dim_branch.", "month_start": "First day of the month.",
  "calendar_year": "Calendar year.", "calendar_month": "Month number 1-12.", "calendar_quarter": "Calendar quarter 1-4.",
  "fiscal_year": "Indian fiscal year (April-March).", "days_in_month": "Days in the month.",
  "units_sold": "Units sold on delivered orders; 0 when none.", "revenue": "Revenue before GST, INR; 0 when none.",
  "order_lines": "Sales lines in the month.", "orders": "Distinct orders in the month.",
  "units_lag_1": "units_sold one month earlier.", "units_lag_12": "units_sold twelve months earlier.",
  "units_roll_mean_3": "Mean units_sold of the previous 3 months (current month excluded)."},
 "feat_demand_variability": {
  "product_id": "FK to dim_product.", "branch_id": "FK to dim_branch.", "months_observed": "Months in the series (72).",
  "total_units": "Total delivered units.", "mean_monthly_units": "Mean monthly units.", "std_monthly_units": "Standard deviation of monthly units.",
  "demand_cv": "std / mean of monthly units (demand variability).", "demand_months": "Months with demand.", "zero_demand_months": "Months with no demand.",
  "zero_demand_share": "zero_demand_months / months_observed.", "adi": "Average demand interval: months_observed / demand_months.",
  "cv2_nonzero": "Squared CV of non-zero monthly demand.",
  "demand_class": "Syntetos-Boylan class: Smooth, Erratic, Intermittent, Lumpy (ADI 1.32, CV2 0.49)."},
}


def _sql(name: str) -> str:
    return (FEATURE_SQL[name]
            .replace("__REF__", REF_DATE)
            .replace("__DSTART__", DEMAND_START)
            .replace("__DEND__", DEMAND_END))


def _esc(text: str) -> str:
    return text.replace("'", "''")


def build_features(con):
    """Create schema `feat`, build all feature tables, attach descriptions.

    Returns a DataFrame: table, rows, columns.
    """
    import pandas as pd
    con.execute("CREATE SCHEMA IF NOT EXISTS feat")
    rows = []
    for name in BUILD_ORDER:
        con.execute(_sql(name))
        con.execute(f"COMMENT ON TABLE feat.{name} IS '{_esc(TABLE_DESCRIPTIONS[name])}'")
        for col, text in COLUMN_DESCRIPTIONS[name].items():
            con.execute(f"COMMENT ON COLUMN feat.{name}.{col} IS '{_esc(text)}'")
        n = con.execute(f"SELECT COUNT(*) FROM feat.{name}").fetchone()[0]
        c = con.execute(f"SELECT COUNT(*) FROM duckdb_columns() "
                        f"WHERE schema_name='feat' AND table_name='{name}'").fetchone()[0]
        rows.append({"table": name, "rows": n, "columns": c})
    return pd.DataFrame(rows)


def undocumented_columns(con):
    """Feature columns that have no description (expected: none)."""
    return con.execute("""
        SELECT table_name, column_name FROM duckdb_columns()
        WHERE schema_name = 'feat' AND comment IS NULL
        ORDER BY table_name, column_index""").df()

Writing src/features.py


**Reading the output.** `Writing src/features.py`. Nothing runs yet; the file is only saved.

## 2. Build all feature tables

**Objective.** Run the library to create schema `feat`.

Tables are built in dependency order: purchase orders before suppliers, sales orders and invoices before customers, monthly demand before demand variability. Re-running is safe (`CREATE OR REPLACE`).

In [3]:
sys.path.insert(0, os.getcwd())
import src.features as features
importlib.reload(features)

build_summary = features.build_features(con)
build_summary

,table,rows,columns
0,feat_purchase_order,24000,19
1,feat_supplier,8,16
2,feat_sales_order,20000,16
3,feat_invoice,18033,20
4,feat_customer,500,19
5,feat_product,30,20
6,feat_monthly_demand,12960,15
7,feat_demand_variability,180,13


**Reading the output.** One row per feature table with its row count and column count. Expected rows: `feat_purchase_order` 24,000 · `feat_supplier` 8 · `feat_sales_order` 20,000 · `feat_invoice` 18,033 · `feat_customer` 500 · `feat_product` 30 · `feat_monthly_demand` 12,960 (30 products x 6 branches x 72 months) · `feat_demand_variability` 180.

## 3. Inspect each feature group

For each group: what it measures, then the output and how to read it.

### 3.1 Purchase-order timing (`feat_purchase_order`)

**Measures.** Planned lead time (expected - order), actual lead time (received - order), **delay days** (received - expected; positive = late) and an **on-time flag**. Cancelled POs have no receipt, so their timing fields are empty by design.

In [4]:
display(con.execute("""
    SELECT po_id, supplier_id, order_date, expected_date, received_date, po_status,
           planned_lead_time_days, actual_lead_time_days, delay_days, is_on_time
    FROM feat.feat_purchase_order
    ORDER BY po_id LIMIT 5
""").df())

con.execute("""
    SELECT po_status, COUNT(*) AS pos,
           ROUND(AVG(delay_days), 2) AS avg_delay_days,
           MIN(delay_days) AS min_delay, MAX(delay_days) AS max_delay,
           ROUND(100.0 * AVG(is_on_time::INT), 1) AS on_time_pct
    FROM feat.feat_purchase_order GROUP BY po_status
""").df()

,po_id,supplier_id,order_date,expected_date,received_date,po_status,planned_lead_time_days,actual_lead_time_days,delay_days,is_on_time
0,PO-100001,SUP0003,2022-12-14,2022-12-30,2022-12-30,Received,16,16,0,True
1,PO-100003,SUP0002,2023-04-21,2023-05-01,2023-05-01,Received,10,10,0,True
2,PO-100013,SUP0005,2023-01-18,2023-02-15,2023-02-17,Received,28,30,2,False
3,PO-100032,SUP0007,2021-01-17,2021-01-29,NaT,Cancelled,12,<NA>,<NA>,<NA>
4,PO-100130,SUP0001,2021-03-09,2021-03-28,2021-03-28,Received,19,19,0,True


,po_status,pos,avg_delay_days,min_delay,max_delay,on_time_pct
0,Received,21630,0.51,-3,7,84.9
1,Cancelled,2370,NaN,<NA>,<NA>,NaN


**Reading the output.** The first table shows sample POs with their derived fields. The second compares Received and Cancelled POs: Received rows should have delay figures and an on-time percentage; Cancelled rows should show empty (NULL) delay and on-time values, confirming cancelled POs stay out of delay measures.

### 3.2 Supplier performance (`feat_supplier`)

**Measures.** For each of the 8 suppliers: **stated vs actual** lead time, lead-time variability, average and 90th-percentile delay, on-time rate, cancellation rate and share of spend. Week 7 builds on this.

In [5]:
con.execute("""
    SELECT supplier_id, supplier_name, stated_lead_time_days,
           ROUND(avg_actual_lead_time_days, 1) AS actual_lead_time,
           ROUND(lead_time_std_days, 1)        AS lead_time_std,
           ROUND(avg_delay_days, 1)            AS avg_delay,
           p90_delay_days,
           ROUND(100 * on_time_rate, 1)        AS on_time_pct,
           ROUND(100 * cancellation_rate, 1)   AS cancel_pct,
           ROUND(100 * spend_share, 1)         AS spend_share_pct,
           stated_reliability_score
    FROM feat.feat_supplier ORDER BY spend_share DESC
""").df()

,supplier_id,supplier_name,stated_lead_time_days,actual_lead_time,lead_time_std,avg_delay,p90_delay_days,on_time_pct,cancel_pct,spend_share_pct,stated_reliability_score
0,SUP0005,Tianjin OEM Supplies,28,28.5,1.6,0.5,2.0,85.3,9.5,12.9,3
1,SUP0001,Shenzhen OEM Supplies,19,19.5,1.7,0.5,3.0,84.9,9.9,12.6,5
2,SUP0008,Hangzhou OEM Supplies,26,26.5,1.7,0.5,3.0,84.2,9.6,12.6,4
3,SUP0003,Shanghai Distributor Supplies,16,16.5,1.8,0.5,3.0,85.0,10.2,12.6,4
4,SUP0006,Qingdao Distributor Supplies,10,10.5,1.7,0.5,3.0,85.2,9.4,12.5,3
5,SUP0007,Suzhou Local Vendor Supplies,12,12.5,1.7,0.5,3.0,84.7,9.9,12.5,5
6,SUP0004,Ningbo Distributor Supplies,18,18.5,1.7,0.5,3.0,84.8,10.2,12.2,3
7,SUP0002,Guangzhou Local Vendor Supplies,10,10.5,1.7,0.5,3.0,85.0,10.3,12.1,5


**Reading the output.** One row per supplier, largest spend first. Compare **stated** lead time and reliability score with the **actual** lead time and on-time percentage: a large gap means the supplier master is not a reliable guide. Spend shares should add up to about 100%.

### 3.3 Sales-order delivery (`feat_sales_order`)

**Measures.** Order totals plus **order-to-delivery days**. For cancelled orders the delivery date is a placeholder (Week 2 finding), so delivery days are left empty.

In [6]:
con.execute("""
    SELECT order_status, COUNT(*) AS orders,
           ROUND(AVG(order_to_delivery_days), 2) AS avg_days_to_deliver,
           MIN(order_to_delivery_days) AS min_days, MAX(order_to_delivery_days) AS max_days,
           ROUND(AVG(revenue_before_gst), 0) AS avg_order_value
    FROM feat.feat_sales_order GROUP BY order_status
""").df()

,order_status,orders,avg_days_to_deliver,min_days,max_days,avg_order_value
0,Delivered,18033,8.06,0,15,1273165.0
1,Cancelled,1967,NaN,<NA>,<NA>,1265076.0


**Reading the output.** Delivered orders show delivery-day statistics (Week 2 found 0 to 15 days). Cancelled orders show empty delivery figures, as intended.

### 3.4 Invoice payment behaviour (`feat_invoice`)

**Measures.** **Days to pay** (fully paid invoices), **late-payment flag**, **overdue flag** and days overdue at the reference date. Ambiguous invoices get no timing because their payments cannot be assigned safely.

In [7]:
con.execute("""
    SELECT payment_status_derived,
           COUNT(*) AS invoices,
           ROUND(AVG(days_to_pay), 1) AS avg_days_to_pay,
           COUNT(*) FILTER (WHERE is_paid_late) AS paid_late,
           COUNT(*) FILTER (WHERE is_overdue)   AS overdue,
           ROUND(AVG(days_overdue), 0)          AS avg_days_overdue
    FROM feat.feat_invoice
    GROUP BY payment_status_derived ORDER BY invoices DESC
""").df()

,payment_status_derived,invoices,avg_days_to_pay,paid_late,overdue,avg_days_overdue
0,Fully paid,12145,32.2,5061,0,NaN
1,Partially paid,3645,NaN,0,3578,1077.0
2,Unpaid,1798,NaN,0,1764,1087.0
3,Ambiguous (duplicate invoice_id),393,NaN,0,0,NaN
4,Overpaid,52,NaN,0,0,NaN


**Reading the output.** Invoices grouped by derived status. Only **Fully paid** rows have days-to-pay and late flags. `Ambiguous` rows should show none of them. Overdue counts appear for Unpaid and Partially paid invoices past their due date at 31 Dec 2024.

### 3.5 Customer behaviour (`feat_customer`)

**Measures.** **Credit utilisation** (balance / credit limit), order value, **recency** (days since last order), tenure and payment behaviour. Week 8 (RFM, churn, CLV) starts from this table.

In [8]:
display(con.execute("""
    SELECT ROUND(AVG(credit_utilisation), 3) AS avg_credit_utilisation,
           ROUND(MEDIAN(credit_utilisation), 3) AS median_credit_utilisation,
           COUNT(*) FILTER (WHERE credit_utilisation > 1) AS customers_over_limit,
           ROUND(AVG(recency_days), 0) AS avg_recency_days,
           COUNT(*) FILTER (WHERE delivered_orders = 0) AS customers_without_orders
    FROM feat.feat_customer
""").df())

con.execute("""
    SELECT customer_id, customer_type, industry_segment, delivered_orders,
           ROUND(revenue_before_gst) AS revenue, ROUND(avg_order_value) AS aov,
           recency_days, ROUND(credit_utilisation, 2) AS credit_util
    FROM feat.feat_customer ORDER BY revenue_before_gst DESC LIMIT 5
""").df()

,avg_credit_utilisation,median_credit_utilisation,customers_over_limit,avg_recency_days,customers_without_orders
0,0.5,0.489,0,61.0,0


,customer_id,customer_type,industry_segment,delivered_orders,revenue,aov,recency_days,credit_util
0,C0153,Retail,Rental,56,84966970.0,1517267.0,19,0.86
1,C0288,Dealer,Infrastructure,55,76291160.0,1387112.0,74,0.07
2,C0483,Corporate,Rental,47,75352940.0,1603254.0,13,0.61
3,C0475,Fleet Owner,Construction,55,74442140.0,1353493.0,13,0.89
4,C0495,Fleet Owner,Infrastructure,48,74165320.0,1545111.0,36,0.33


**Reading the output.** The first table summarises credit use and recency across all 500 customers (a value above 1 means a customer is over its credit limit). The second lists the 5 highest-revenue customers with their order value and recency.

### 3.6 Product attributes (`feat_product`)

**Measures.** Unit margin, margin % and markup %, unit volume (m3), weight, density and price per kg. Week 6 uses volume for space utilisation.

In [9]:
con.execute("""
    SELECT product_id, product_name, unit_cost, unit_price,
           ROUND(margin_pct_calc, 1) AS margin_pct, margin_pct_stored,
           ROUND(volume_m3, 4) AS volume_m3, weight_kg
    FROM feat.feat_product ORDER BY product_id LIMIT 10
""").df()

,product_id,product_name,unit_cost,unit_price,margin_pct,margin_pct_stored,volume_m3,weight_kg
0,P001,Hydraulic Pump HP-300,18950,24500,22.7,29.2,0.0250,38.5
1,P002,Engine Oil Filter OF-90,450,720,37.5,37.3,0.0006,0.4
2,P003,Air Filter AF-120,820,1250,34.4,52.4,0.0026,0.8
3,P004,Track Chain TC-45,32500,39800,18.3,22.5,0.2520,210.0
4,P005,Fuel Injector FI-220,6350,8450,24.9,33.0,0.0029,1.2
5,P006,Alternator ALT-500,8650,11200,22.8,29.5,0.0120,9.5
6,P007,Hydraulic Hose HH-60,980,1450,32.4,48.0,0.0014,1.1
7,P008,Bucket Tooth BT-10,340,580,41.4,70.6,0.0003,0.3
8,P009,Starter Motor SM-700,11450,14900,23.2,30.1,0.0104,11.0
9,P010,Turbocharger TB-900,28900,34900,17.2,20.8,0.0250,16.5


**Reading the output.** The first 10 of 30 products. Compare `margin_pct` (recalculated) with `margin_pct_stored`: if they differ a lot, the stored field uses a different definition and the recalculated one should be used for KPIs.

### 3.7 Monthly demand (`feat_monthly_demand`)

**Measures.** Delivered units per **product x branch x month**, with months of zero demand filled in (so variability is measured correctly), calendar fields and lag features for forecasting in Week 9.

In [10]:
display(con.execute("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT product_id) AS products,
           COUNT(DISTINCT branch_id) AS branches, COUNT(DISTINCT month_start) AS months,
           MIN(month_start) AS first_month, MAX(month_start) AS last_month
    FROM feat.feat_monthly_demand
""").df())

con.execute("""
    SELECT product_id, branch_id, month_start, units_sold, units_lag_1, units_lag_12, units_roll_mean_3
    FROM feat.feat_monthly_demand
    WHERE product_id = (SELECT MIN(product_id) FROM feat.feat_monthly_demand)
      AND branch_id  = (SELECT MIN(branch_id)  FROM feat.feat_monthly_demand)
    ORDER BY month_start LIMIT 14
""").df()

,rows,products,branches,months,first_month,last_month
0,12960,30,6,72,2019-01-01,2024-12-01


,product_id,branch_id,month_start,units_sold,units_lag_1,units_lag_12,units_roll_mean_3
0,P001,AHM001,2019-01-01,48.0,NaN,NaN,NaN
1,P001,AHM001,2019-02-01,46.0,48.0,NaN,48.000000
2,P001,AHM001,2019-03-01,121.0,46.0,NaN,47.000000
3,P001,AHM001,2019-04-01,99.0,121.0,NaN,71.666667
4,P001,AHM001,2019-05-01,82.0,99.0,NaN,88.666667
5,P001,AHM001,2019-06-01,118.0,82.0,NaN,100.666667
6,P001,AHM001,2019-07-01,116.0,118.0,NaN,99.666667
7,P001,AHM001,2019-08-01,134.0,116.0,NaN,105.333333
8,P001,AHM001,2019-09-01,88.0,134.0,NaN,122.666667
9,P001,AHM001,2019-10-01,142.0,88.0,NaN,112.666667


**Reading the output.** The first table should show 12,960 rows = 30 products x 6 branches x 72 months (Jan 2019 - Dec 2024). The second shows one product-branch series: lag features are empty for the first months (no earlier data), and `units_lag_12` only starts in month 13.

### 3.8 Demand variability (`feat_demand_variability`)

**Measures.** **Demand CV** (std / mean of monthly units) and the **demand pattern**, using the standard Syntetos-Boylan rule: average demand interval (ADI, threshold 1.32) and squared CV of non-zero demand (threshold 0.49) give **Smooth, Erratic, Intermittent or Lumpy**. Week 4 (ABC-XYZ) and Week 9 (forecast method per class) use it.

In [11]:
display(con.execute("""
    SELECT demand_class, COUNT(*) AS product_branch_pairs,
           ROUND(AVG(demand_cv), 2) AS avg_cv,
           ROUND(AVG(adi), 2) AS avg_adi,
           ROUND(AVG(mean_monthly_units), 1) AS avg_monthly_units
    FROM feat.feat_demand_variability
    GROUP BY demand_class ORDER BY product_branch_pairs DESC
""").df())

con.execute("""
    SELECT ROUND(MIN(demand_cv),2) AS min_cv, ROUND(MEDIAN(demand_cv),2) AS median_cv,
           ROUND(MAX(demand_cv),2) AS max_cv
    FROM feat.feat_demand_variability
""").df()

,demand_class,product_branch_pairs,avg_cv,avg_adi,avg_monthly_units
0,Smooth,180,0.41,1.0,95.3


,min_cv,median_cv,max_cv
0,0.31,0.41,0.54


**Reading the output.** The first table counts the 180 product-branch pairs by demand pattern. Mostly **Smooth** means regular, predictable demand; **Intermittent / Lumpy** means many zero months and harder forecasting. The second shows the spread of CV; the closer to 0, the steadier the demand.

## 4. Validate the features

**Objective.** Prove the features are complete and follow the rules. Each check compares an expected value with an actual value; all should be `PASSED`.

In [12]:
checks = [
 ("rows: feat_purchase_order", "SELECT COUNT(*) FROM (SELECT DISTINCT po_id FROM star.fact_purchase_line)", "SELECT COUNT(*) FROM feat.feat_purchase_order"),
 ("rows: feat_sales_order",    "SELECT COUNT(*) FROM (SELECT DISTINCT so_id FROM star.fact_sales_line)",    "SELECT COUNT(*) FROM feat.feat_sales_order"),
 ("rows: feat_invoice",        "SELECT COUNT(*) FROM star.fact_invoice_payment", "SELECT COUNT(*) FROM feat.feat_invoice"),
 ("rows: feat_customer",       "SELECT COUNT(*) FROM star.dim_customer", "SELECT COUNT(*) FROM feat.feat_customer"),
 ("rows: feat_product",        "SELECT COUNT(*) FROM star.dim_product",  "SELECT COUNT(*) FROM feat.feat_product"),
 ("rows: feat_supplier",       "SELECT COUNT(*) FROM star.dim_supplier", "SELECT COUNT(*) FROM feat.feat_supplier"),
 ("rows: feat_monthly_demand = 30 x 6 x 72", "SELECT 30*6*72", "SELECT COUNT(*) FROM feat.feat_monthly_demand"),
 ("rows: feat_demand_variability = 180",     "SELECT 180",      "SELECT COUNT(*) FROM feat.feat_demand_variability"),
 ("total: demand units = delivered units",
    "SELECT SUM(quantity) FROM star.fact_sales_line WHERE NOT is_cancelled",
    "SELECT SUM(units_sold) FROM feat.feat_monthly_demand"),
 ("total: customer revenue = delivered revenue",
    "SELECT SUM(line_total) FROM star.fact_sales_line WHERE NOT is_cancelled",
    "SELECT SUM(revenue_before_gst) FROM feat.feat_customer"),
 ("rule: cancelled POs have no delay", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_purchase_order WHERE is_cancelled AND delay_days IS NOT NULL"),
 ("rule: received POs all have a delay", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_purchase_order WHERE NOT is_cancelled AND delay_days IS NULL"),
 ("rule: cancelled orders have no delivery days", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_sales_order WHERE is_cancelled AND order_to_delivery_days IS NOT NULL"),
 ("rule: ambiguous invoices have no days_to_pay", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_invoice WHERE invoice_id_duplicated AND days_to_pay IS NOT NULL"),
 ("rule: every customer has a credit utilisation", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_customer WHERE credit_utilisation IS NULL"),
 ("rule: every product has a volume", "SELECT 0", "SELECT COUNT(*) FROM feat.feat_product WHERE volume_m3 IS NULL"),
]
rows = []
for name, exp_sql, act_sql in checks:
    exp = float(con.execute(exp_sql).fetchone()[0])
    act = float(con.execute(act_sql).fetchone()[0])
    rows.append((name, exp, act, "PASSED" if abs(exp - act) < 0.01 else "FAILED"))
feature_checks = pd.DataFrame(rows, columns=["check", "expected", "actual", "status"])
display(feature_checks)
print(f"{(feature_checks.status == 'PASSED').sum()} of {len(feature_checks)} checks passed.")

,check,expected,actual,status
0,rows: feat_purchase_order,2.400000e+04,2.400000e+04,PASSED
1,rows: feat_sales_order,2.000000e+04,2.000000e+04,PASSED
2,rows: feat_invoice,1.803300e+04,1.803300e+04,PASSED
3,rows: feat_customer,5.000000e+02,5.000000e+02,PASSED
4,rows: feat_product,3.000000e+01,3.000000e+01,PASSED
5,rows: feat_supplier,8.000000e+00,8.000000e+00,PASSED
6,rows: feat_monthly_demand = 30 x 6 x 72,1.296000e+04,1.296000e+04,PASSED
7,rows: feat_demand_variability = 180,1.800000e+02,1.800000e+02,PASSED
8,total: demand units = delivered units,1.234939e+06,1.234939e+06,PASSED
9,total: customer revenue = delivered revenue,2.295899e+10,2.295899e+10,PASSED


16 of 16 checks passed.


**Reading the output.** One row per check. `rows:` and `total:` checks confirm no data was lost or duplicated; `rule:` checks confirm the Week 2 decisions (cancelled and ambiguous records) were honoured. All should say `PASSED`. A `FAILED` row names the problem: send me that row and I will trace it.

In [13]:
assert (feature_checks.status == "PASSED").all(), "A feature check failed - see the table above."
features.undocumented_columns(con)

,table_name,column_name


**Reading the output.** The assertion stops the notebook if any check failed. The table below it lists feature columns with **no description**; it should be **empty**, meaning every table and column is documented (these descriptions feed the data dictionary, A6).

## 5. Summary

| Item | Result |
|---|---|
| Library | `src/features.py`, rebuildable with `build_features(con)` |
| Tables | 8 feature tables in schema `feat`, all described in the database |
| Validation | Row counts, totals and Week 2 rules checked |
| Rules honoured | Cancelled orders/POs excluded from timing features; ambiguous invoices excluded from payment timing |

**Status: COMPLETED** once all checks pass and the undocumented list is empty.

**Hand-off.** `03_kpi_baseline.ipynb` reads these tables to compute the baseline KPIs.

In [14]:
con.close()
print("Connection closed.")

Connection closed.
